# 3つのMP4を横並びの比較動画にする

`outputs/` の実行フォルダから3本を指定し、左・中央・右に並べた1本のMP4を作ります。

- 3本の先頭を時刻0にそろえて同時再生します。動作のタイミングを自動で合わせる処理はありません。同じ区間から作った素材を使ってください。
- 高さをそろえて3本を隙間なく横に並べ、全体を1920×1080（16:9）に収めます。縦横比を維持し、必要な黒い余白は外側だけに付けます。長さは最短の動画に合わせます。入力映像自体にある余白は除去しません。
- 出力は無音のH.264 MP4です。FPSを統一する際はフレームが間引き・複製されます。
- 姿勢推定や追跡は再実行しません。骨格・追跡枠は推定結果であり、動きの正解を示すものではありません。

プロジェクト直下で `uv sync`、`uv run jupyter lab compare_three_videos.ipynb` を実行し、上から順にセルを実行してください。FFmpegとffprobeが必要です（macOS: `brew install ffmpeg`）。


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import subprocess
import uuid

from IPython.display import Video, display

ROOT = Path.cwd()
OUTPUT_ROOT = ROOT / "outputs"
if not OUTPUT_ROOT.is_dir():
    raise RuntimeError("outputs/ があるプロジェクト直下で起動してください。")
for executable in ("ffmpeg", "ffprobe"):
    if shutil.which(executable) is None:
        raise RuntimeError(f"{executable} が必要です。FFmpegをインストールしてください。")

print("入力フォルダ候補（.working.mp4 は処理途中のため除外）:")
for folder in sorted(OUTPUT_ROOT.iterdir()):
    if folder.is_dir():
        names = sorted(p.name for p in folder.glob("*.mp4") if not p.name.endswith(".working.mp4"))
        if names:
            print(folder.name, names)


## 1. 入力を指定

`SOURCE_DIR` を上に表示されたフォルダ名に変更してください。`VIDEO_FILES` の順番が左・中央・右です。別フォルダの動画を使う場合は、各項目に絶対パスを指定できます。

`WITH_TRACKING` で追跡付き素材を選択できます。現在の入力フォルダ・ファイル指定は次のセルを確認してください。
`OUTPUT_WIDTH`・`OUTPUT_HEIGHT` は完成動画全体のサイズ（正の偶数）です。初期値は1920×1080（16:9）。個々の映像に余白は追加せず、全体を中央に配置します。端数の丸めにより縦横比に最大数ピクセル程度の差が生じます。


In [ ]:
WITH_TRACKING = True
WHO = "トミタヒカル"

OUTPUT_WIDTH = 1920
OUTPUT_HEIGHT = 1080
OUTPUT_FPS = 30


if WITH_TRACKING:
    SOURCE_DIR = OUTPUT_ROOT / f"{WHO}_tracking付き"
    VIDEO_FILES = ["skeleton_tracking.mp4", "original.mp4", "pose_tracking.mp4"]
else:
    SOURCE_DIR = OUTPUT_ROOT / f"{WHO}_trackingなし"
    VIDEO_FILES = ["skeleton.mp4", "original.mp4", "overlay.mp4"]


## 2. 合成して保存

`outputs/comparisons/日時_ID/` に `comparison.mp4` と設定・入力情報の `manifest.json` を保存します。再実行ごとに新しいフォルダを作ります。
処理時間は動画の長さに依存します。成功すると保存先と出力情報を表示します。


In [ ]:
if len(VIDEO_FILES) != 3:
    raise ValueError("VIDEO_FILES にはMP4をちょうど3つ指定してください。")
if any(type(value) is not int or value <= 0 or value % 2 for value in (OUTPUT_WIDTH, OUTPUT_HEIGHT)):
    raise ValueError("出力全体の幅と高さは正の偶数にしてください。")
if type(OUTPUT_FPS) is not int or not 1 <= OUTPUT_FPS <= 120:
    raise ValueError("OUTPUT_FPS は1〜120の整数にしてください。")

input_paths = [(SOURCE_DIR / Path(name)).resolve() for name in VIDEO_FILES]
for path in input_paths:
    if not path.is_file() or path.suffix.lower() != ".mp4":
        raise ValueError(f"MP4が見つかりません: {path}")
    if path.name.endswith(".working.mp4"):
        raise ValueError(f"処理途中の素材は使えません: {path}")

def probe_video(path):
    result = subprocess.run([
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=width,height,avg_frame_rate,duration:format=duration",
        "-of", "json", str(path),
    ], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"動画情報の取得失敗: {path}\n{result.stderr}")
    info = json.loads(result.stdout)
    if not info.get("streams"):
        raise ValueError(f"映像ストリームがありません: {path}")
    return info

input_info = [probe_video(path) for path in input_paths]
for position, path, info in zip(("左", "中央", "右"), input_paths, input_info):
    print(position, path.name, info)

# 共通の高さにそろえ、表示上の縦横比（dar）から各映像の幅を決める。
# 各映像にはpadを入れず、連結後の全体にだけ余白を付ける。
filters = []
for index in range(3):
    filters.append(
        f"[{index}:v:0]setpts=PTS-STARTPTS,fps={OUTPUT_FPS},"
        f"scale=w='max(2,trunc({OUTPUT_HEIGHT}*dar/2)*2)':h={OUTPUT_HEIGHT},"
        f"setsar=1,format=yuv420p[v{index}]"
    )
filters.append(
    "[v0][v1][v2]hstack=inputs=3:shortest=1,"
    f"scale={OUTPUT_WIDTH}:{OUTPUT_HEIGHT}:force_original_aspect_ratio=decrease:force_divisible_by=2,"
    f"setsar=1,pad={OUTPUT_WIDTH}:{OUTPUT_HEIGHT}:(ow-iw)/2:(oh-ih)/2:black[out]"
)

OUTPUT_DIR = OUTPUT_ROOT / "comparisons" / (datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8])
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
OUTPUT_VIDEO = OUTPUT_DIR / "comparison.mp4"
working_video = OUTPUT_DIR / "comparison.working.mp4"
command = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-n"]
for path in input_paths:
    command.extend(["-i", str(path)])
command.extend([
    "-filter_complex", ";".join(filters), "-map", "[out]", "-an",
    "-c:v", "libx264", "-preset", "medium", "-crf", "18",
    "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(working_video),
])
print("合成中…")
result = subprocess.run(command, capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(f"合成失敗（作業ファイル: {working_video}）\n{result.stderr}")
output_info = probe_video(working_video)
stream = output_info["streams"][0]
if (stream["width"], stream["height"]) != (OUTPUT_WIDTH, OUTPUT_HEIGHT):
    raise RuntimeError("出力サイズが設定と一致しません。")
working_video.rename(OUTPUT_VIDEO)
manifest = {
    "inputs_left_to_right": [str(path) for path in input_paths],
    "input_info": input_info,
    "output_size": [OUTPUT_WIDTH, OUTPUT_HEIGHT], "fps": OUTPUT_FPS,
    "layout": "equal height, no gaps between inputs, outer padding only",
    "alignment": "each input starts at zero; stop at shortest input",
    "audio": False, "output": str(OUTPUT_VIDEO.resolve()),
    "output_info": output_info, "ffmpeg_command": command,
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("保存先:", OUTPUT_VIDEO)
print("出力情報:", output_info)
